# 09. Итоговая таблица вакансий 2024–2026

`vacancies_2024_2026.csv` (ноутбук 07) собран для кода: 37 технических колонок, вспомогательные поля, разные стили названий.
Здесь из него строится **итоговая таблица для чтения**: понятные названия, логичный порядок, только нужные колонки.

**Результат:**
- `data/processed/vacancies_final.csv`: все колонки, включая текст объявления;
- `data/processed/vacancies_final.xlsx`: лист «вакансии» (без длинных текстов) и лист «словарь» с описанием колонок.

Что меняется по сравнению с исходным файлом:
- только вакансии с **2024 года** (старые объявления lalafo 2017–2023 не входят);
- опыт, который не указан в объявлении, остаётся **пустым** (в файле для моделей он заменён на 1 год);
- добавлены уровень из заголовка (junior…lead), формат работы из поля «Тип» Telegram-постов и отметка повторной публикации;
- статус зарплаты одной колонкой: указана / скрыта / неправдоподобная.

In [1]:
import json, re
import numpy as np
import pandas as pd

src = pd.read_csv("../data/processed/vacancies_2024_2026.csv", low_memory=False, parse_dates=["published"])
d = src[src["published"] >= "2024-01-01"].copy()
print(f"Исходный файл: {len(src)} строк, {src.shape[1]} колонок | с 2024 года: {len(d)}")

Исходный файл: 44656 строк, 37 колонок | с 2024 года: 41302


## Формат работы из поля «Тип» постов codifynews

In [2]:
typ = {}
for line in open("../data/raw/telegram_posts.jsonl", encoding="utf-8"):
    r = json.loads(line)
    m = re.search(r"Тип:\s*(.+)", r.get("text") or "")
    if m:
        typ[r["link"]] = m.group(1).strip().lower()
raw_type = d["link"].map(typ)

def work_format(t, remote):
    t = str(t) if isinstance(t, str) else ""
    if re.search(r"стаж|intern", t): return "стажировка"
    if "project" in t or "проект" in t: return "проектная работа"
    if "частичн" in t or "неполный" in t or "part" in t: return "частичная занятость"
    if re.search(r"remote|удал", t) or remote: return "удалённо или гибрид"
    if "office" in t or "полная" in t or "полный" in t: return "офис, полный день"
    return "не указан"

d["work_format"] = [work_format(t, r) for t, r in zip(raw_type, d["remote"])]
d["work_format"].value_counts()

work_format
офис, полный день      20370
не указан              13282
удалённо или гибрид     6809
стажировка               335
частичная занятость      299
проектная работа         207
Name: count, dtype: int64

## Уровень из заголовка и повторные публикации

In [3]:
LEVELS = [("lead", r"\b(lead|тимлид|team ?lead|ведущ\w*|head|руководител\w*|principal|architect|архитектор)\b"),
          ("senior", r"\b(senior|сеньор|синьор|старш\w*|sr)\b"),
          ("middle", r"\b(middle|мидл|mid)\b"),
          ("junior", r"\b(junior|джун\w*|младш\w*|jr|стаж[её]р\w*|intern|trainee|ассистент)\b")]
def level(t):
    t = str(t).lower()
    for n, p in LEVELS:
        if re.search(p, t):
            return n
    return "не указан"
d["level"] = d["name"].map(level)

norm = lambda s: s.fillna("").str.lower().str.replace(r"[^а-яёa-z0-9]+", " ", regex=True).str.strip()
key = norm(d["name"]) + "|" + norm(d["company"]) + "|" + norm(d["requirements"]).str.slice(0, 100)
d = d.sort_values("published")
gap = d.assign(k=key, s=d["salary_kgs"].round(-2)).groupby(["k", "s"])["published"].diff()
d["is_repost"] = (gap <= pd.Timedelta(days=30)) & d["has_salary"]
print(f"Повторных публикаций с той же зарплатой за 30 дней: {d['is_repost'].sum()}")

Повторных публикаций с той же зарплатой за 30 дней: 119


## Сборка итоговой таблицы

In [4]:
SOURCE = {"headhunter": "hh.kg", "lalafo": "lalafo.kg", "telegram_codifynews": "Telegram @codifynews",
          "telegram_kgworkzone": "Telegram @kgworkzone"}
EXPERIENCE = {"нет опыта": "без опыта", "1–3 года": "1–3 года", "3–6 лет": "3–6 лет", "6+ лет": "6+ лет", "не указан": "не указан"}
STATUS = np.select([d["has_salary"], d["salary_outlier"]], ["указана", "неправдоподобная, удалена"], "скрыта")
blank = lambda s: s.replace({"не указан": np.nan, "не указано": np.nan, "": np.nan})

final = pd.DataFrame({
    "vacancy_id": d["id"],
    "published_date": d["published"].dt.date,
    "year": d["published"].dt.year,
    "month": d["published"].dt.to_period("M").astype(str),
    "source": d["source"].map(SOURCE),
    "post_url": d["link"],
    "job_title": d["name"].str.strip(),
    "employer": blank(d["company"]),
    "sector": d["category"],
    "profession": d["sub_category"],
    "profession_predicted": d["category_predicted"].fillna(False).astype(bool),
    "level": d["level"],
    "city_group": blank(d["city"]),
    "locality": blank(d["area"]),
    "experience_required": d["experience"].map(EXPERIENCE).fillna("не указан"),
    "experience_min_years": d["experience_start"].where(d["experience_known"].fillna(True).astype(bool)),
    "work_format": d["work_format"],
    "education_required": blank(d["education"]),
    "gender_required": blank(d["gender_req"]),
    "schedule": blank(d["schedule"]),
    "commission_or_bonus": d["commission"].astype(bool),
    "salary_status": STATUS,
    "salary_from": d["compensation_from"],
    "salary_to": d["compensation_to"],
    "currency": d["compensation_currency"],
    "pay_period": d["pay_period"].map({"month": "месяц", "day": "день", "week": "неделя", "hour": "час"}),
    "salary_month_kgs": d["salary_kgs"].round(-1),
    "is_repost": d["is_repost"],
    "skills": d["skills"],
    "description": (d["requirements"].fillna("") + "\n" + d["responsibilities"].fillna("")).str.strip().replace("", np.nan),
}).sort_values(["published_date", "vacancy_id"]).reset_index(drop=True)
final["experience_min_years"] = final["experience_min_years"].astype("Int64")
print(f"Итог: {len(final)} строк, {final.shape[1]} колонок")
final.head(3).T

Итог: 41302 строк, 30 колонок


,0,1,2
vacancy_id,105438465,53981188,58580588
published_date,2024-01-01,2024-01-01,2024-01-01
year,2024,2024,2024
month,2024-01,2024-01,2024-01
source,lalafo.kg,lalafo.kg,lalafo.kg
post_url,https://lalafo.kg/sokuluk/ads/cac-taracka-univ...,https://lalafo.kg/bishkek/ads/torgovyj-agent-s...,https://lalafo.kg/bishkek/ads/srocno-trebuetsa...
job_title,Парикмахер Универсал. Процент,Торговый агент. С личным транспортом,Продавец-консультант
employer,NaN,NaN,NaN
sector,"Спортивные клубы, фитнес, салоны красоты","Продажи, обслуживание клиентов",Розничная торговля
profession,Парикмахер,Торговый представитель,"Продавец-консультант, продавец-кассир"


## Словарь данных

In [5]:
DICT = [
    ("vacancy_id", "идентификатор вакансии", "tg_codifynews_17354_0"),
    ("published_date", "дата публикации", "2024-07-30"),
    ("year", "год публикации", "2024"),
    ("month", "месяц публикации", "2024-07"),
    ("source", "источник: hh.kg, lalafo.kg, Telegram @codifynews, Telegram @kgworkzone", "Telegram @codifynews"),
    ("post_url", "ссылка на объявление или пост", "https://t.me/codifynews/17354"),
    ("job_title", "название вакансии из объявления", "Site Reliability Engineer"),
    ("employer", "работодатель (на lalafo и kgworkzone почти всегда пусто)", "T-Bank"),
    ("sector", "сфера (27 значений)", "Информационные технологии"),
    ("profession", "профессия (около 170 значений)", "Программист, разработчик"),
    ("profession_predicted", "сфера и профессия определены классификатором по заголовку (Telegram), а не источником", "True"),
    ("level", "уровень из заголовка: junior / middle / senior / lead / не указан", "senior"),
    ("city_group", "Бишкек / Ош / Регионы; пусто, если не указан", "Бишкек"),
    ("locality", "населённый пункт (в основном hh и lalafo)", "Бишкек"),
    ("experience_required", "требуемый опыт: без опыта / 1–3 года / 3–6 лет / 6+ лет / не указан", "1–3 года"),
    ("experience_min_years", "нижняя граница опыта в годах: 0, 1, 3, 6; пусто, если не указан", "1"),
    ("work_format", "офис, полный день / удалённо или гибрид / частичная занятость / проектная работа / стажировка / не указан", "удалённо или гибрид"),
    ("education_required", "требование к образованию из текста", "высшее"),
    ("gender_required", "требование к полу из текста", "женщины"),
    ("schedule", "график из текста (5/2, 2/2 …)", "5/2"),
    ("commission_or_bonus", "в оплате есть проценты или бонусы", "False"),
    ("salary_status", "указана / скрыта / неправдоподобная, удалена", "указана"),
    ("salary_from", "зарплата «от» как в объявлении (в исходной валюте и периоде)", "40000"),
    ("salary_to", "зарплата «до» как в объявлении", "50000"),
    ("currency", "валюта объявления: KGS / USD / EUR", "KGS"),
    ("pay_period", "период ставки: месяц / день / неделя / час", "месяц"),
    ("salary_month_kgs", "итоговая зарплата: сом в месяц, по курсу НБКР; пусто, если скрыта или неправдоподобна", "45000"),
    ("is_repost", "повтор той же вакансии с той же зарплатой в течение 30 дней (в моделях удаляется)", "False"),
    ("skills", "ключевые навыки (почти только hh)", "Python, SQL"),
    ("description", "текст требований и обязанностей", "…"),
]
dictionary = pd.DataFrame(DICT, columns=["колонка", "описание", "пример"])
assert list(dictionary["колонка"]) == list(final.columns)
dictionary

,колонка,описание,пример
0,vacancy_id,идентификатор вакансии,tg_codifynews_17354_0
1,published_date,дата публикации,2024-07-30
2,year,год публикации,2024
3,month,месяц публикации,2024-07
4,source,"источник: hh.kg, lalafo.kg, Telegram @codifyne...",Telegram @codifynews
5,post_url,ссылка на объявление или пост,https://t.me/codifynews/17354
6,job_title,название вакансии из объявления,Site Reliability Engineer
7,employer,работодатель (на lalafo и kgworkzone почти все...,T-Bank
8,sector,сфера (27 значений),Информационные технологии
9,profession,профессия (около 170 значений),"Программист, разработчик"


## Проверка и сохранение

In [6]:
summary = final.groupby(["source", "year"]).agg(вакансий=("vacancy_id", "size"),
                                                 с_зарплатой=("salary_month_kgs", "count"),
                                                 медиана=("salary_month_kgs", "median"))
display(summary.style.format({"медиана": "{:,.0f}"}))

final.to_csv("../data/processed/vacancies_final.csv", index=False)

xl = final.drop(columns=["description", "skills"])
with pd.ExcelWriter("../data/processed/vacancies_final.xlsx", engine="openpyxl") as w:
    xl.to_excel(w, sheet_name="вакансии", index=False)
    dictionary.to_excel(w, sheet_name="словарь", index=False)
    summary.reset_index().to_excel(w, sheet_name="сводка", index=False)
    from openpyxl.styles import Font, PatternFill
    from openpyxl.utils import get_column_letter
    for name, frame in [("вакансии", xl), ("словарь", dictionary), ("сводка", summary.reset_index())]:
        ws = w.sheets[name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for i, col in enumerate(frame.columns, 1):
            ws.cell(1, i).font = Font(bold=True, color="FFFFFF")
            ws.cell(1, i).fill = PatternFill("solid", fgColor="3B7DD8")
            width = min(max(len(str(col)), frame[col].astype(str).str.len().quantile(0.9) if len(frame) else 10) + 2, 60)
            ws.column_dimensions[get_column_letter(i)].width = width
print("Сохранено: data/processed/vacancies_final.csv и vacancies_final.xlsx")

Сохранено: data/processed/vacancies_final.csv и vacancies_final.xlsx


## Выводы

1. **Итоговая таблица:** 41 302 вакансии с января 2024 по октябрь 2026, 30 понятных колонок вместо 37 технических.
2. **Открывается в Excel** (`vacancies_final.xlsx`): закреплённая шапка, фильтры, лист «словарь» с описанием каждой колонки
   и лист «сводка» по источникам и годам.
3. **Файл для моделей не меняется:** `vacancies_2024_2026.csv` остаётся входом ноутбука 08 и репозитория kg-salary-prediction.